In [1]:

# ============================================================
# 🚀 PIPELINE FINAL: METABOLISMO + CLÍNICO + KM (VERSIÓN 2026)
# ============================================================

# 1. CARGA DE LIBRERÍAS
required_libs <- c("dplyr", "ggplot2", "reshape2", "ggpubr", "survival",
                   "survminer", "readxl", "purrr", "tidyr", "grid", "gridExtra",
                   "scales")
invisible(lapply(required_libs, library, character.only = TRUE))

# ===============================
# 2. CONFIGURACIÓN DE RUTAS
# ===============================
paths <- list(
  metabolic    = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Metabolic data analysis/ML models using metabolic data/resultados_de_pareto_UMAP_conruido/Merged_TCGA_BRCA_AllData_safe_withSelectedClusters_PARETO_con_ruido.csv",
  clinicaldata = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Clinical data analysis/ML models using clinical data/Results_clustering_UMAP/DATA_MASTER_CLUSTERS_Y_CLINICA.csv",
  div          = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_divergentes_para_estudio_pyn.csv",
  core         = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_core_correlacion_Paretoynormas.csv"
)

group_colors <- c("Core" = "#0072B2", "Divergent" = "#D55E00")

# ─── TEMA GLOBAL ────────────────────────────────────────────────────────────
base_theme <- theme_pubr() +
  theme(
    title         = element_text(size = 18, face = "bold"),
    axis.title    = element_text(size = 15),
    axis.text     = element_text(size = 13),
    legend.text   = element_text(size = 13),
    legend.title  = element_text(size = 14, face = "bold"),
    strip.text    = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 13, color = "grey50")
  )

# ===============================
# 3. PREPROCESAMIENTO Y CRUCE
# ===============================
clean_names <- function(x) toupper(gsub("\\.", "-", trimws(as.character(x))))

df_metab <- read.csv(paths$metabolic,    stringsAsFactors = FALSE)
df_clin  <- read.csv(paths$clinicaldata, stringsAsFactors = FALSE)
df_div   <- read.csv(paths$div,          stringsAsFactors = FALSE)
df_core  <- read.csv(paths$core,         stringsAsFactors = FALSE)

df_metab$ModelName <- clean_names(df_metab$ModelName)
df_clin$ModelName  <- clean_names(df_clin$ModelName)
div_names          <- clean_names(df_div$ModelName)
core_names         <- clean_names(df_core$ModelName)

# ─── FIX 1: Detectar colisiones entre grupos ────────────────────────────────
overlap <- intersect(div_names, core_names)
if (length(overlap) > 0) {
  warning(paste0("⚠️  ", length(overlap), " pacientes en ambos grupos (se excluirán): ",
                 paste(head(overlap, 5), collapse = ", ")))
}

analysis <- df_metab %>%
  mutate(Group = case_when(
    ModelName %in% overlap    ~ NA_character_,
    ModelName %in% div_names  ~ "Divergent",
    ModelName %in% core_names ~ "Core",
    TRUE                      ~ "Other"
  )) %>%
  filter(Group %in% c("Core", "Divergent"))

# ─── FIX 2: Join clínico ────────────────────────────────────────────────────
# OS y OS.time vienen de df_metab; el join trae SOLO variables clínicas descriptivas
clinical_cols <- c(
  "ModelName",
  # Estadificación tumoral
  "ajcc_pathologic_stage.diagnoses",
  "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses",
  "ajcc_pathologic_m.diagnoses",
  "tumor_grade.diagnoses",
  "morphology.diagnoses",
  "primary_diagnosis.diagnoses",
  # Tratamiento
  "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses",
  "prior_treatment.diagnoses",
  # Muestra / tejido
  "sample_type.samples",
  "tissue_type.samples",
  "tumor_descriptor.samples",
  "specimen_type.samples",
  "is_ffpe.samples",
  "oct_embedded.samples",
  # Fisiología del paciente
  "age_at_diagnosis.diagnoses",
  "alcohol_history.exposures",
  # Molecular / metadata
  "Menopausal.Status",       # pueden venir con punto en lugar de espacio tras read.csv
  "Cancer.Type",
  "ER",
  "PR",
  "HER2",
  "Subtype",
  "Genetic.Ancestry",
  # Flags calculados
  "Prior_Treatment_Flag",
  "Metastasis_Flag",
  "ER_PR_HER2_Combo",
  "ER_PR_HER2_Combo_encoded",
  "Subtype_encoded"
)

clin_subset <- df_clin %>%
  select(any_of(clinical_cols))

# Eliminar columnas clínicas que pudieran existir en df_metab para evitar duplicados
cols_from_clin <- setdiff(colnames(clin_subset), "ModelName")
analysis <- analysis %>%
  select(-any_of(cols_from_clin)) %>%
  left_join(clin_subset, by = "ModelName")

# ─── Renombrar columnas con punto a espacio (si read.csv los convirtió) ─────
analysis <- analysis %>%
  rename_with(~ gsub("Menopausal\\.Status", "Menopausal Status", .x)) %>%
  rename_with(~ gsub("Cancer\\.Type",       "Cancer Type",       .x)) %>%
  rename_with(~ gsub("Genetic\\.Ancestry",  "Genetic Ancestry",  .x))

# ===============================
# 4. ANÁLISIS ESTADÍSTICO (FDR)
# ===============================
run_stats <- function(data, columns) {
  results <- map_df(columns, function(col) {
    sub_data <- data %>%
      select(Group, value = !!sym(col)) %>%
      filter(is.finite(value), !is.na(value))

    group_sizes <- table(sub_data$Group)
    if (length(group_sizes) < 2 || any(group_sizes < 3)) return(NULL)

    test <- wilcox.test(value ~ Group, data = sub_data, exact = FALSE)

    meds <- sub_data %>%
      group_by(Group) %>%
      summarise(m = median(value), .groups = "drop")

    data.frame(
      Feature  = col,
      Med_Core = meds$m[meds$Group == "Core"],
      Med_Div  = meds$m[meds$Group == "Divergent"],
      p_val    = test$p.value
    )
  })

  if (nrow(results) == 0) return(results)

  results %>%
    mutate(p_adj = p.adjust(p_val, method = "fdr")) %>%
    arrange(p_val)
}

metab_res  <- run_stats(analysis, grep("SA_|Ratio",  colnames(analysis), value = TRUE))
pareto_res <- run_stats(analysis, grep("^Pareto_",   colnames(analysis), value = TRUE))

# Variables numéricas clínicas para test estadístico
numeric_clinical_cols <- c("age_at_diagnosis.diagnoses", "is_ffpe.samples",
                           "oct_embedded.samples", "Prior_Treatment_Flag",
                           "Metastasis_Flag", "ER_PR_HER2_Combo_encoded", "Subtype_encoded")
numeric_clinical_cols <- intersect(numeric_clinical_cols, colnames(analysis))
clinical_num_res <- run_stats(analysis, numeric_clinical_cols)

# ===============================
# 5. FUNCIONES DE PLOTEO
# ===============================

# --- Violin + Boxplot para features cuantitativos ---
plot_box_feat <- function(feat, data, stats_df) {
  p_info <- stats_df %>% filter(Feature == feat)
  if (nrow(p_info) == 0) return(NULL)
  p_info <- p_info %>% slice(1)

  plot_data <- data %>% filter(is.finite(!!sym(feat)), !is.na(!!sym(feat)))
  if (nrow(plot_data) == 0) return(NULL)

  ggplot(plot_data, aes(x = Group, y = !!sym(feat), fill = Group)) +
    geom_violin(alpha = 0.2, color = NA) +
    geom_boxplot(width = 0.4, outlier.shape = 16, outlier.size = 2) +
    labs(title = feat, subtitle = paste("p-adj:", signif(p_info$p_adj, 3)), y = NULL) +
    scale_fill_manual(values = group_colors) +
    base_theme +
    theme(legend.position = "none")
}

# --- Variables clínicas: boxplot si numérica, barras si categórica ---
plot_clinical <- function(var, data) {
  if (!var %in% colnames(data)) return(NULL)
  tmp <- data %>% filter(!is.na(!!sym(var)))
  if (nrow(tmp) == 0) return(NULL)

  # Limpiar etiquetas largas para el eje X
  tmp[[var]] <- trimws(as.character(tmp[[var]]))

  if (is.numeric(tmp[[var]])) {
    ggplot(tmp, aes(x = Group, y = !!sym(var), fill = Group)) +
      geom_boxplot(outlier.shape = 16, outlier.size = 2) +
      stat_compare_means(label = "p.signif", label.size = 5) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, x = "") +
      base_theme
  } else {
    # Calcular chi-squared p-value para categóricas
    tbl <- table(tmp[[var]], tmp$Group)
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    subtitle_txt <- if (!is.na(p_chi)) paste("Chi² p:", signif(p_chi, 3)) else ""

    ggplot(tmp, aes(x = !!sym(var), fill = Group)) +
      geom_bar(position = "fill") +
      scale_y_continuous(labels = percent) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, subtitle = subtitle_txt, y = "Proporción") +
      base_theme +
      theme(axis.text.x = element_text(angle = 40, hjust = 1, size = 10))
  }
}

# ===============================
# 6. GENERACIÓN DEL REPORTE PDF
# ===============================
pdf("Reporte_Metabolismo_Clinico_2026.pdf", width = 14, height = 11)

# ─── Página de título ────────────────────────────────────────────────────────
grid.newpage()
grid.text("Reporte: Core vs Divergent\nMetabolismo + Clínica + Supervivencia",
          gp = gpar(fontsize = 26, fontface = "bold"), y = 0.55)
grid.text(paste("Pacientes Core:", sum(analysis$Group == "Core"),
                "  |  Divergentes:", sum(analysis$Group == "Divergent")),
          gp = gpar(fontsize = 16), y = 0.40)

# ─── Metabolismo ────────────────────────────────────────────────────────────
if (nrow(metab_res) > 0) {
  metab_plots <- map(head(metab_res$Feature, 6), ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)
  if (length(metab_plots) > 0)
    grid.arrange(grobs = metab_plots, ncol = 3, nrow = 2,
                 top = textGrob("Top Metabolitos Diferenciales",
                                gp = gpar(fontsize = 20, fontface = "bold")))
}

# ─── Pareto ─────────────────────────────────────────────────────────────────
if (nrow(pareto_res) > 0) {
  pareto_plots <- map(head(pareto_res$Feature, 6), ~plot_box_feat(.x, analysis, pareto_res)) %>%
    Filter(Negate(is.null), .)
  if (length(pareto_plots) > 0)
    grid.arrange(grobs = pareto_plots, ncol = 3, nrow = 2,
                 top = textGrob("Top Métricas Pareto",
                                gp = gpar(fontsize = 20, fontface = "bold")))
}

# ─── Variables Clínicas Numéricas ───────────────────────────────────────────
if (nrow(clinical_num_res) > 0) {
  num_plots <- map(clinical_num_res$Feature, ~plot_box_feat(.x, analysis, clinical_num_res)) %>%
    Filter(Negate(is.null), .)
  if (length(num_plots) > 0) {
    for (i in seq(1, length(num_plots), by = 6)) {
      idx <- i:min(i + 5, length(num_plots))
      grid.arrange(grobs = num_plots[idx], ncol = 3,
                   top = textGrob("Variables Clínicas Numéricas",
                                  gp = gpar(fontsize = 20, fontface = "bold")))
    }
  }
}

# ─── Variables Clínicas Categóricas ─────────────────────────────────────────
# Lista completa de variables categóricas a analizar
clinical_to_plot <- c(
  "ajcc_pathologic_stage.diagnoses",
  "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses",
  "ajcc_pathologic_m.diagnoses",
  "tumor_grade.diagnoses",
  "morphology.diagnoses",
  "primary_diagnosis.diagnoses",
  "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses",
  "prior_treatment.diagnoses",
  "sample_type.samples",
  "tissue_type.samples",
  "tumor_descriptor.samples",
  "specimen_type.samples",
  "alcohol_history.exposures",
  "Menopausal Status",
  "Cancer Type",
  "ER",
  "PR",
  "HER2",
  "Subtype",
  "Genetic Ancestry",
  "ER_PR_HER2_Combo"
)

# Solo plotear las que realmente existen en analysis
clinical_to_plot <- intersect(clinical_to_plot, colnames(analysis))

c_plots <- map(clinical_to_plot, ~plot_clinical(.x, analysis)) %>%
  Filter(Negate(is.null), .)

# Imprimir de a 2 por página
if (length(c_plots) > 0) {
  for (i in seq(1, length(c_plots), by = 2)) {
    idx <- i:min(i + 1, length(c_plots))
    grid.arrange(grobs = c_plots[idx], nrow = length(idx), ncol = 1,
                 top = textGrob("Distribución Clínica por Grupo: Core vs Divergent",
                                gp = gpar(fontsize = 20, fontface = "bold")))
  }
}

# ─── Kaplan-Meier ────────────────────────────────────────────────────────────
if (all(c("OS.time", "OS") %in% colnames(analysis))) {
  km_data <- analysis %>%
    filter(!is.na(OS.time), !is.na(OS), OS.time > 0)

  if (nrow(km_data) > 0 && length(unique(km_data$Group)) == 2) {
    fit <- survfit(Surv(OS.time, OS) ~ Group, data = km_data)
    print(ggsurvplot(fit, data = km_data,
                     pval           = TRUE,
                     pval.size      = 5,
                     conf.int       = TRUE,
                     palette        = unname(group_colors),
                     risk.table     = TRUE,
                     risk.table.cex = 1.0,
                     surv.line.size = 1.2,
                     title          = "Curva de Supervivencia: Core vs Divergent",
                     xlab           = "Tiempo (días)",
                     ylab           = "Probabilidad de Supervivencia",
                     legend.labs    = c("Core", "Divergent"),
                     theme          = base_theme +
                       theme(plot.title  = element_text(size = 22, face = "bold"),
                             axis.title  = element_text(size = 16),
                             axis.text   = element_text(size = 14),
                             legend.text = element_text(size = 14))))
  } else {
    warning("⚠️  Datos insuficientes para Kaplan-Meier tras filtrado.")
  }
} else {
  warning("⚠️  Columnas OS y/o OS.time no encontradas en el dataset metabólico.")
}

dev.off()
cat("\n✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'\n")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘survminer’


The following object is masked from ‘package:survival’:

    myeloma



Attaching package: ‘tidyr’


The following object is masked from ‘package:reshape2’:

    smiths



Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine



Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


Warning message in chisq.test(tbl):
“Chi-squared approximation may be incorrect”
Warning message in chisq.test(tbl):
“Chi-squared approximation may be incorrect”
Warning message in chisq.test(tbl):
“Chi-squared approximation may be incorrect”
Warning message in chisq.test(tbl):
“Chi-squared approximation may be incorrect”
Warning message in chisq.test(tbl):
“Chi-squared approximation may be

pdf 
  2


✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'


In [8]:
install.packages("pheatmap")

Installing package into ‘/Users/eduardoruiz/Library/R/arm64/4.4/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/x0/j78w4f8n0_z4l575_k16n2k40000gn/T//RtmpPiHpQZ/downloaded_packages
